# Native hybrid subset-majority QNN

Train the full original data split with the reviewed checkpoint and 4096 shots per quantum evaluation. Run from the repository root. This local workflow retains one executed epoch, not the helper default of 20 or a commented 30.


In [ ]:
from matplotlib import pyplot as plt

import flagquantum as fq
from examples.synthesis.algorithms.machine_learning.hybrid_qnn.hybrid_qnn_for_subset_majority.main import (
    build_program,
)
from examples.synthesis.algorithms.machine_learning.hybrid_qnn.hybrid_qnn_for_subset_majority.overview.workflow import (
    SHOTS,
    load_network,
    make_dataset,
    plot_workflow,
    run_workflow,
)


## Data and checkpoint

Classical 10-to-4 linear/Tanh feeds two native quantum qubits, then a 2-to-1 linear/ReLU head. Labels use the strict majority of indices 1, 3, 4, 6, 7, 9. Original tensor values are portable JSON, represented locally as float64.


In [ ]:
data = make_dataset()
network = load_network()
print("Training/test shapes:", data.training.shape, data.test.shape)
print("Checkpoint quantum weights:", network.quantum_weight.detach())


## Native quantum program

The parent native qfunc allocates two qubits, encodes four RX/RZ features, and applies two RY-equivalent rotations and RZZ. Named readout removes manual bit-string decoding. The next cell inspects a bound program; training uses actual sampled P(1) values.


In [ ]:
program = build_program([0.2, -0.3, 0.4, -0.5], [0.1, 0.2, 0.3])
result = fq.run(program)
print(dict(program.probabilities(result, registers=("result",))))
print("Per-circuit resources:", program.report["resources"])


## One sampled training epoch and held-out evaluation

Four feature derivatives propagate back to the first classical layer; all three quantum weights also receive parameter-shift gradients. Five full batches and one partial batch train all 340 examples. The 512 test examples are disjoint. A reproducible local seed adapts the source's unspecified Torch/shot seeds; source accuracy and training trajectory are not asserted. This is an explicit gradient bridge, not a generic differentiable qfunc layer.


In [ ]:
workflow = run_workflow(seed=17)
for index, step in enumerate(workflow.steps, start=1):
    print("Minibatch/size/MSE:", index, step.size, step.loss)
print("Held-out accuracy:", workflow.accuracy)
print("Quantum evaluations/shots:", workflow.quantum_evaluations, SHOTS)
print("Final bound resources:", workflow.program.report["resources"])


## Actual observations

The plots use all six measured minibatch losses and the sampled predictions for all 512 held-out examples. The workload is 5612 quantum evaluations, each drawing 4096 joint outcomes, including shifted executions. This local educational example makes no hardware, scalability or quantum-advantage claim.


In [ ]:
figure = plot_workflow(workflow)
plt.show()
figure
